# U-ADAPT — Supervisor Demo Visualizations

**Mode A uncertainty-gated fusion — publication-quality evidence of the research contribution.**

This notebook renders the six demo figures from the outputs of
`scripts/demo_mode_a_end_to_end.py`. If the results do not exist yet, the
first cell runs the demo pipeline automatically (deterministic, seed=0,
synthetic world or real cache).

| # | Figure | What it proves |
|---|--------|----------------|
| 1 | Gate weight distribution | the gate is **dynamic** (not stuck at w=0.5) |
| 2 | Uncertainty–accuracy (D1/D2) | uncertainty proxies **predict error** (core assumption) |
| 3 | Gate favorability (D3) | the gate **chooses the better modality** > 50% |
| 4 | Gap recovery | U-ADAPT **recovers part of the zero-shot→transfer gap** |
| 5 | Qualitative examples | gate behavior on real (or synthetic) proposals |
| 6 | Coefficient ablation | each gate component (alpha/beta/gamma) contributes |

**Validation checklist:** deterministic (seed=0) · figures render · runs in
<30 min on Colab · supervisor-ready.

**One-click Colab run:** run the first code cell below — it clones the
repo, installs requirements (skipping the CUDA-preinstalled torch), and
runs the full demo on a free T4. No local setup needed.

In [ ]:
# --- ONE-CLICK COLAB SETUP: clone repo, install deps, run the demo --------
# Run this cell FIRST on Colab (free T4). It:
#   1. clones this repository into /content/u-adapt-disaster-perception (if absent)
#   2. installs requirements.txt (skipping torch/torchvision — preinstalled
#      with CUDA on Colab; see the requirements.txt note)
#   3. runs the full demo -> outputs/supervisor_demo/{results,proposal_level}.json
# Locally (or when the repo is already mounted) it is a no-op.
import os
import subprocess
import sys
from pathlib import Path

REPO_URL = "https://github.com/toufiq-dev/u-adapt-disaster-perception.git"
REPO_DIR = Path("/content/u-adapt-disaster-perception")
COLAB = "google.colab" in sys.modules or os.environ.get("COLAB_GPU") is not None

if not COLAB:
    print("[setup] not on Colab — skipping clone/install (local run).")
else:
    if not (REPO_DIR / "scripts" / "demo_mode_a_end_to_end.py").exists():
        print("[setup] cloning repo ...")
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, str(REPO_DIR)], check=True)
    else:
        # Refresh an existing clone (fast-forward only; ignore failures).
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=False)
    os.chdir(REPO_DIR)
    sys.path.insert(0, str(REPO_DIR / "src"))
    print(f"[setup] repo at {REPO_DIR}")

    # Install requirements, skipping torch/torchvision (CUDA-preinstalled).
    req = (REPO_DIR / "requirements.txt").read_text().splitlines()
    pkgs = [
        l.split()[0]
        for l in req
        if l.strip() and not l.lstrip().startswith("#")
        and not l.split()[0].lower().startswith("torch")
    ]
    print(f"[setup] installing {len(pkgs)} packages (torch/torchvision skipped) ...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", *pkgs], check=True)

    # Run the whole demo (deterministic seed=0; synthetic unless a real cache exists).
    print("[setup] running the demo ...")
    subprocess.run(
        [sys.executable, "scripts/demo_mode_a_end_to_end.py",
         "--out", "outputs/supervisor_demo/results.json",
         "--proposal-out", "outputs/supervisor_demo/proposal_level.json"],
        check=True,
    )
    print("[setup] demo done — run the cells below to render Figures 1-6.")


In [ ]:
# --- setup: run demo if results missing, then load ------------------------
import json
import os
import subprocess
import sys
from pathlib import Path

# locate repo root: walk up from cwd and check known Colab mount paths
def _find_root():
    candidates = [Path.cwd(), *Path.cwd().parents, Path("/content/u-adapt-disaster-perception")]
    for cand in candidates:
        if (cand / "scripts" / "demo_mode_a_end_to_end.py").exists():
            return cand
    return Path.cwd()

ROOT = _find_root()
os.chdir(ROOT)
sys.path.insert(0, str(ROOT / "src"))

OUT = ROOT / "outputs" / "supervisor_demo"
FIG = OUT / "figures"
FIG.mkdir(parents=True, exist_ok=True)
RESULTS = OUT / "results.json"
PROPOSALS = OUT / "proposal_level.json"

if not RESULTS.exists():
    print("results.json not found — running demo pipeline (seed=0)...")
    subprocess.run(
        [sys.executable, "scripts/demo_mode_a_end_to_end.py",
         "--out", str(RESULTS), "--proposal-out", str(PROPOSALS)],
        cwd=ROOT, check=True,
    )

results = json.loads(RESULTS.read_text())
proposal_payload = json.loads(PROPOSALS.read_text())
proposals = proposal_payload["proposals"]
ground_truth = proposal_payload["ground_truth"]

import matplotlib.pyplot as plt
import numpy as np
try:
    import seaborn as sns
    sns.set_theme(style="whitegrid")
except ImportError:
    pass  # seaborn optional; plain matplotlib styling is fine

from uadapt.demo.plotting import (
    figure1_gate_weights,
    figure2_d1_d2,
    figure3_gate_favorability,
    figure4_gap_recovery,
    figure5_qualitative,
    figure6_ablation,
)

print("data source:", results["meta"].get("data_source"))
print("mAP50:", {k: round(v, 4) for k, v in results["map50"].items()})

## Figure 1 — Gate Weight Distribution

**Claim:** the gate is *dynamic* — it does not collapse to naive averaging
(w = 0.5). A wide spread of w proves the analytic rule is responding to
per-proposal uncertainty (σ²_text, σ²_visual, affinity). Color shows which
modality was correct: if the gate works, text-correct proposals should skew
toward low w and visual-correct proposals toward high w.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
note = figure1_gate_weights(proposals, ax)
fig.tight_layout()
fig.savefig(FIG / "figure1_gate_weights.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

### Methodological Note — Pooling Requirement (deviation 2026-08-03)

D1/D2/D3 are structurally underpowered on D-Fire alone: 2 classes (fire,
smoke) yield only 2 distinct variance values, from which no meaningful
Spearman ρ or gate-favorability trend can be computed. Per pre-registration
deviation **§10**, D1/D2/D3 are therefore evaluated **pooled across
LADD+D-Fire** (3 distinct classes → 3 distinct variance values) for the
primary diagnostic claim; per-dataset values are still reported. See
`docs/pre_registration.md` and `docs/change_log.md` (2026-08-03, 2026-08-04).
The figures below show 6-class demo values; **real-data figures will
supersede these synthetic demo figures.**


## Figure 2 — Uncertainty–Accuracy Correlation (D1/D2)

**Claim (core assumption):** higher normalized uncertainty ⇒ higher error
rate. Binned scatter of error rate vs σ²_text (D1) and σ²_visual (D2) with
Spearman ρ — the *validity* of the whole gating mechanism rests on these.
Values shown are from the 6-class demo; for real data the primary claim is
reported **pooled across LADD+D-Fire** (see methodological note above,
deviation 2026-08-03).


In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(11, 4.5))
note = figure2_d1_d2(results["diagnostics"], axes)
fig.tight_layout()
fig.savefig(FIG / "figure2_d1_d2.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

### Methodological Note — Synthetic-World D3 Artifact

The synthetic demo world was implicitly engineered around the min-max
normalization stretch. Under the mathematically correct absolute scaling
(x/2.0), the raw variance magnitudes are small relative to the affinity term,
which depresses D3 (to ≈5.4% on the 2-class synthetic stand-in) — a
**demo-world artifact, not a methodological flaw**. Real data will have
different variance magnitudes. Diagnostic **D5** is the pre-registered
sentinel: if real variances cluster near 0 or 1, it flags and triggers the
pre-registered Beta-regression fallback. See
`docs/supervisor_demo_report.md` (Methodological Caveats) and
`docs/change_log.md` (2026-08-03). **Real-data figures will supersede these
synthetic demo figures.**


## Figure 3 — Gate Favorability (D3)

**Claim (is the gate useful?):** among proposals where the two modalities
*disagree*, the gate assigns higher weight to the more accurate modality
significantly more often than chance (binomial test vs 50%). For real data
the primary claim is reported **pooled across LADD+D-Fire** (deviation
2026-08-03). On the synthetic stand-in under absolute scaling, D3 is
depressed by the demo-world artifact (see methodological note above).


In [ ]:
fig, ax = plt.subplots(figsize=(6.5, 4.5))
note = figure3_gate_favorability(results["diagnostics"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure3_gate_favorability.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Figure 4 — Gap Recovery Analysis

**Claim (RQ2):** U-ADAPT recovers a fraction of the zero-shot→transfer gap:

$$\text{gap recovery} = \frac{mAP_{U-ADAPT} - mAP_{zero}}{mAP_{oracle} - mAP_{zero}} \times 100\%$$

Bars: zero-shot (raw scores) → U-ADAPT Mode A → transfer ceiling (oracle
re-rank: every GT-correct proposal ranked above every incorrect one — the
maximum any re-scoring method can reach on this proposal set). Dashed lines
(if configured) show the literature zero-shot / transfer mAP50 from the
dataset config — **dataset-specific references (e.g. D-Fire 27.5 → 65.6),
not a comparable baseline** to the demo subset.

In [ ]:
fig, ax = plt.subplots(figsize=(8, 4.5))
note = figure4_gap_recovery(results["gap_recovery"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure4_gap_recovery.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)
print("gap recovery payload:", {k: round(v, 4) if isinstance(v, float) else v
                                 for k, v in results["gap_recovery"].items()})

## Figure 5 — Qualitative Examples

Three panels:

1. **High w** — the gate trusted *visual* (low visual uncertainty / high affinity);
2. **Low w** — the gate trusted *text* (high visual uncertainty / low affinity);
3. **Gate corrected naive averaging** — the two modalities disagreed and the
   gate picked the correct one, where w=0.5 averaging would have diluted it.

When the demo ran on **real cached features** (Milestone 1+), the panels show
the **real detection images** with GT (green) and proposal (blue) boxes. In
synthetic mode (no imagery cached yet) they fall back to a schematic scene —
the wiring is identical either way.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 4.5))
image_paths = proposal_payload.get("image_paths")
note = figure5_qualitative(proposals, ground_truth, axes, seed=0, image_paths=image_paths)
fig.tight_layout()
fig.savefig(FIG / "figure5_qualitative.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)
print("real-image panels enabled:", bool(image_paths))

## Figure 6 — Ablation Study

**Claim (component contributions):** removing a term changes gate behavior.
On the demo subset, **alpha (visual uncertainty) contributes** (alpha=0 drops
mAP50); beta (text uncertainty) and gamma (affinity) effects are small /
within noise — expected on this synthetic world, where the gating signal is
dominated by the visual-uncertainty term. The figure reports the numbers
honestly; significance testing is deferred to the real-data protocol (§9).

In [ ]:
fig, ax = plt.subplots(figsize=(7.5, 4.5))
note = figure6_ablation(results["ablation"], ax)
fig.tight_layout()
fig.savefig(FIG / "figure6_ablation.png", dpi=150, bbox_inches="tight")
plt.show()
print("annotation:", note)

## Summary & Next Steps

Figures are saved to `outputs/supervisor_demo/figures/`. The 2-page
supervisor report is at `docs/supervisor_demo_report.md` (embeds these
figures and interprets the D1–D3 diagnostics).

**Next steps before thesis submission:**
- Re-run on real cached features (Milestone 1+): `--cache-dir cached_features
  --ground-truth data/annotations/dfire_test.json --norm-strategy absolute`
  (absolute scaling, x/2.0, is required for the 2-class D-Fire run — min-max
  collapses the variance terms to {0, 1}: on the stand-in, min-max gives
  0.906 mAP50 (loses to naive 0.955), while absolute restores it to 0.956
  (beats naive));
- 10-seed statistical protocol (paired t-test + Wilcoxon, §9);
- full mAP50:95 + ECE/Brier/uncertainty-AUROC via `scripts/04_evaluate.py`;
- cross-backbone ablation (OWL-ViT, YOLOE26) for RQ5.

*Synthetic-data caveat: the numbers above are a mechanism demonstration.
They validate the wiring and the diagnostics — not a research result.*
*Methodological caveats (deviation 2026-08-03): D1/D2/D3 on D-Fire alone
are structurally underpowered (2 classes → 2 distinct variance values) and
are evaluated pooled across LADD+D-Fire; the synthetic world was engineered
around the min-max stretch, so the raw variance magnitudes are small vs the
affinity term under absolute scaling (a demo-world artifact) — D5 is the
pre-registered sentinel triggering the Beta-regression fallback if real
variances cluster near 0. See docs/change_log.md.*
